# Tutorial 3: Data Analysis & Products\n
In this notebook, we explore the Zarr products using `xarray`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from panoseti_analysis.io.stores import open_store

## Load L0 (Raw Counts)

In [ ]:
l0_store_path = "results_tutorial/L0/obs_TEST.pffd.dp_img16.bpp_2.module_1.debug_TRUNCATED.zarr"
ds_l0 = open_store(l0_store_path)
display(ds_l0)

## Load L1 (Median Subtracted)

In [ ]:
l1_store_path = "results_tutorial/L1/obs_TEST.dp_img16.module_1.L1.zarr"
ds_l1 = open_store(l1_store_path)
display(ds_l1)

## The `ds.pano` data contract\n\nEvery PANOSETI Dataset has a `.pano` accessor that gives you a typed interface to the canonical attributes.  It is registered automatically when you import from `panoseti_analysis.config`.

In [ ]:
# ds.pano gives typed access to canonical attributes
print(f"level : {ds_l1.pano.level!r}")  # ds.attrs["data_level"]
print(f"kind  : {ds_l1.pano.kind!r}")  # inferred from data_product

# validate() checks the schema contract and returns ds (chainable)
ds_l1.pano.validate(level="L1", kind=ds_l1.pano.kind)
print("✓ L1/img contract satisfied")

## QC report\n\nEvery L1 store carries a `qc` attrs block stamped during calibration.  No extra I/O needed — it rode in with the Zarr attrs.

In [ ]:
qc = ds_l1.attrs.get("qc", {})
print(f"isgood  : {qc.get('isgood')}")
print(f"metrics : {qc.get('metrics')}")
print()
for check in qc.get("checks", []):
    status = "✓" if check["passed"] else "✗"
    print(
        f"  {status} {check['key']:<25} value={check['value']:.4f}  threshold={check['threshold']:.4f}"
    )

In [ ]:
# Calibration provenance (resolution key present when --recipe was used)
cal = ds_l1.attrs.get("calibration", {})
print(f"Calibration kind : {cal.get('kind')}")
resolution = cal.get("resolution")
if resolution:
    print(f"Recipe hash      : {resolution['source_hash']}")
    print(f"Params           : {resolution['params']}")
else:
    print("(no calibration recipe was used — defaults applied)")

In [ ]:
plt.figure(figsize=(6, 5))
plt.imshow(ds_l1["median_subtracted"].values[0], cmap="viridis")
plt.colorbar()
plt.title("L1: First Frame")
plt.show()

## Load L2 (Cloud Detection Scores)

In [ ]:
l2_store_path = "results_tutorial/L2/obs_TEST.cloud.module_1.zarr"
ds_l2 = open_store(l2_store_path)
display(ds_l2)

In [ ]:
plt.plot(ds_l2["cloud_score"].values)
plt.title("Cloud Score")
plt.xlabel("Window Index")
plt.ylabel("Score")
plt.show()

## Cloud Detector Step-by-Step\n
Walkthrough of the internal mechanisms of the cloud detector.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from panoseti_analysis.io.models import load_classifier
from panoseti_analysis.paths import MODELS

## Load Model

In [ ]:
model, bundle = load_classifier(MODELS / "cloud_detector_v1.pt")
model.eval()
print("Model architecture:", model)

## Feature Extraction (Hann Window + FFT)

In [ ]:
ds_l2 = xr.open_zarr("results_tutorial/L2/obs_TEST.cloud.module_1.zarr", consolidated=False)
display(ds_l2)
plt.imshow(np.log(ds_l2["feature_deriv_fft"][0] + 1e-7))

In [ ]:
from panoseti_analysis.io.stores import open_store

ds_l1 = open_store("results_tutorial/L1/obs_TEST.dp_img16.module_1.L1.zarr")

# Validate the data contract before doing any science — raises immediately if a
# required variable is missing rather than producing a cryptic downstream error.
ds_l1.pano.validate(level="L1", kind="img")
print(f"✓ level={ds_l1.pano.level!r}  kind={ds_l1.pano.kind!r}")

# Quick-check the QC report stamped during calibration
qc = ds_l1.attrs.get("qc", {})
if qc:
    print(f"  QC isgood={qc['isgood']}  metrics={qc['metrics']}")

img = ds_l1["median_subtracted"].values

# Get a single frame
frame = img[0]
hann_2d = np.outer(np.hanning(32), np.hanning(32))

fft = np.abs(np.fft.fftn(frame * hann_2d))
fft = np.log(np.fft.fftshift(fft))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.imshow(frame)
ax1.set_title("median_subtracted (L1 frame 0)")
ax2.imshow(fft)
ax2.set_title("Hann-windowed FFT")
plt.show()